In [2]:
import findspark
findspark.init()

In [3]:
from  pyspark.sql import SparkSession

In [4]:
spark = SparkSession.builder \
        .appName("Hello world spark") \
        .master("spark://spark-master:7077") \
        .config("spark.ui.port", "4040") \
        .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/19 00:23:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [51]:
from  pyspark.sql.types import StructField, StructType, StringType, IntegerType

data = [
    (1, "Alice", 29),
    (1, "Bob", 35),
    (3, "Charlie", 41)
]


In [58]:
from  pyspark.sql.types import StructField, StructType, StringType, TimestampType, FloatType
from datetime import datetime

factory_data = [
    ("M001", datetime(2026, 1, 26, 8, 0, 0), 75.3),
    ("M002", datetime(2026, 1, 26, 8, 5, 0), 68.7),
    ("M001", datetime(2026, 1, 26, 8, 10, 0), 76.1),
    ("M003", datetime(2026, 1, 26, 8, 15, 0), 72.4),
    ("M002", datetime(2026, 1, 26, 8, 20, 0), 69.8),
    ("M001", datetime(2026, 1, 26, 8, 25, 0), 77.5),
    ("M003", datetime(2026, 1, 26, 8, 30, 0), 73.2),
    ("M002", datetime(2026, 1, 26, 8, 35, 0), 70.1),
    ("M001", datetime(2026, 1, 26, 8, 40, 0), 78.0),
    ("M003", datetime(2026, 1, 26, 8, 45, 0), 74.6),
]

factory_schema = StructType([
    StructField("machine_id", StringType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("temp", FloatType(), True)
])


In [59]:
factory_schema

StructType([StructField('machine_id', StringType(), True), StructField('timestamp', TimestampType(), True), StructField('temp', FloatType(), True)])

In [60]:
factory_df = spark.createDataFrame(factory_data, schema=factory_schema)
factory_df.printSchema()

root
 |-- machine_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- temp: float (nullable = true)



In [61]:
factory_df.columns

['machine_id', 'timestamp', 'temp']

In [62]:
factory_df.printSchema()

root
 |-- machine_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- temp: float (nullable = true)



In [63]:
from pyspark.sql.functions import col, avg, max, min, count, desc

avg_temp = factory_df.groupBy("machine_id") \
                     .agg(avg("temp").alias("avg_temp"))
avg_temp.show()

[Stage 25:======================================>                   (2 + 1) / 3]

+----------+-----------------+
|machine_id|         avg_temp|
+----------+-----------------+
|      M002|69.53333282470703|
|      M003|73.39999898274739|
|      M001|76.72500038146973|
+----------+-----------------+



In [64]:
factory_df = factory_df.orderBy(col("temp"), ascending=False)
factory_df.show()

+----------+-------------------+----+
|machine_id|          timestamp|temp|
+----------+-------------------+----+
|      M001|2026-01-26 08:40:00|78.0|
|      M001|2026-01-26 08:25:00|77.5|
|      M001|2026-01-26 08:10:00|76.1|
|      M001|2026-01-26 08:00:00|75.3|
|      M003|2026-01-26 08:45:00|74.6|
|      M003|2026-01-26 08:30:00|73.2|
|      M003|2026-01-26 08:15:00|72.4|
|      M002|2026-01-26 08:35:00|70.1|
|      M002|2026-01-26 08:20:00|69.8|
|      M002|2026-01-26 08:05:00|68.7|
+----------+-------------------+----+



In [67]:
min_max = factory_df.groupBy("machine_id").agg(
    max("temp").alias("max"),
    min("temp").alias("min")
)

min_max.show()

[Stage 29:>                                                         (0 + 3) / 3]

+----------+----+----+
|machine_id| max| min|
+----------+----+----+
|      M002|70.1|68.7|
|      M001|78.0|75.3|
|      M003|74.6|72.4|
+----------+----+----+



In [68]:
high_temp = factory_df.filter(factory_df.temp > 75)
high_temp.show()

+----------+-------------------+----+
|machine_id|          timestamp|temp|
+----------+-------------------+----+
|      M001|2026-01-26 08:40:00|78.0|
|      M001|2026-01-26 08:25:00|77.5|
|      M001|2026-01-26 08:10:00|76.1|
|      M001|2026-01-26 08:00:00|75.3|
+----------+-------------------+----+



In [69]:
readings = factory_df.groupBy("machine_id") \
                           .agg(count("*").alias("readings"))
readings.show()

+----------+--------+
|machine_id|readings|
+----------+--------+
|      M002|       3|
|      M003|       3|
|      M001|       4|
+----------+--------+

